# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
GRID = np.logspace(-5, 0, 6)
MAX_EPOCHS = 400
TOL = 1e-9
BASE_LR = 0.1  

def loss(y_true, y_pred):
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

class LinearOptimiser:
    """Линейная регрессия с пятью вариантами градиентного метода."""
    def __init__(self, method, lr=None, decay=None, max_epochs=MAX_EPOCHS,
                 tol=TOL, random_state=RANDOM_STATE):
        self.method, self.lr, self.decay = method, lr, decay
        self.max_epochs, self.tol = max_epochs, tol
        self.rng = np.random.default_rng(random_state)

    def _step_size(self, epoch):
        return self.lr if self.decay is None else BASE_LR / (1 + self.decay * epoch)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, p = X.shape
        Xb = np.c_[np.ones(n), X]
        w = np.zeros(p + 1)
        velocity = np.zeros_like(w)
        m = np.zeros_like(w)
        v = np.zeros_like(w)
        grad_memory = np.zeros((n, p + 1)) if self.method == 'SAG' else None
        avg_grad = np.zeros(p + 1)
        previous = np.inf

        for epoch in range(1, self.max_epochs + 1):
            eta = self._step_size(epoch - 1)
            if self.method in ('VGD', 'Momentum', 'Adam'):
                grad = Xb.T @ (Xb @ w - y) / n
                if self.method == 'VGD':
                    w -= eta * grad
                elif self.method == 'Momentum':
                    velocity = 0.9 * velocity + grad
                    w -= eta * velocity
                else:
                    m = 0.9 * m + 0.1 * grad
                    v = 0.999 * v + 0.001 * grad**2
                    m_hat = m / (1 - 0.9**epoch)
                    v_hat = v / (1 - 0.999**epoch)
                    w -= eta * m_hat / (np.sqrt(v_hat) + 1e-8)
            elif self.method == 'SGD':
                for i in self.rng.permutation(n):
                    grad_i = Xb[i] * (Xb[i] @ w - y[i])
                    w -= eta * grad_i
            elif self.method == 'SAG':
                for i in self.rng.permutation(n):
                    grad_i = Xb[i] * (Xb[i] @ w - y[i])
                    avg_grad += (grad_i - grad_memory[i]) / n
                    grad_memory[i] = grad_i
                    w -= eta * avg_grad

            current = np.mean((Xb @ w - y)**2)
            if not np.isfinite(current):
                break
            if abs(previous - current) < self.tol:
                break
            previous = current
        self.coef_, self.n_iter_ = w, epoch
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.c_[np.ones(len(X)), X] @ self.coef_

def metrics(model, X, y_scaled, y_original):
    prediction = model.predict(X) * y_std + y_mean
    return {'Loss': loss(y_original, prediction), 'R2': r2_score(y_original, prediction)}

def run_search(method, schedule):
    records = []
    for value in GRID:
        model = LinearOptimiser(method, lr=value if schedule == 'constant' else None,
                                decay=value if schedule == 'decay' else None).fit(X_train, y_train)
        train = metrics(model, X_train, y_train, y_train_raw)
        val = metrics(model, X_val, y_val, y_val_raw)
        records.append({'parameter': value, 'Loss_train': train['Loss'],
                        'R2_train': train['R2'], 'Loss_val': val['Loss'],
                        'iterations': model.n_iter_})
    search = pd.DataFrame(records)
    best = search.loc[search['Loss_val'].idxmin()]
    best_value = float(best['parameter'])
    final_model = LinearOptimiser(method, lr=best_value if schedule == 'constant' else None,
                                  decay=best_value if schedule == 'decay' else None).fit(X_train, y_train)
    test = metrics(final_model, X_test, y_test, y_test_raw)
    label = f'{method}, ' + (f'eta={best_value:.2e}' if schedule == 'constant'
                               else f'eta(t)={BASE_LR}/(1+{best_value:.2e}*t)')
    result = {'Метод': label, 'Лучший параметр': best_value,
              'Loss_train': best['Loss_train'], 'Loss_val': best['Loss_val'],
              'Loss_test': test['Loss'], 'R2_train': best['R2_train'],
              'R2_test': test['R2'], 'Итераций': final_model.n_iter_}
    return search, result

all_results = []
def show_experiment(method, schedule):
    search, result = run_search(method, schedule)
    all_results.append(result)
    display(search)
    print('Выбран параметр:', f"{result['Лучший параметр']:.2e}")
    print(f"Train: Loss={result['Loss_train']:,.0f}, R²={result['R2_train']:.4f}; "
          f"Val loss={result['Loss_val']:,.0f}; Test: Loss={result['Loss_test']:,.0f}, "
          f"R²={result['R2_test']:.4f}; итераций={result['Итераций']}")
df = pd.read_csv('auto_dataset.csv')
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
display(df.head())

Размер датасета: 1000 строк, 10 столбцов


,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [2]:
target = 'price'
X_raw = df.drop(columns=target)
y_raw = df[target].to_numpy()
cat_cols = X_raw.select_dtypes(include='object').columns.tolist()
num_cols = X_raw.select_dtypes(exclude='object').columns.tolist()
print('Категориальные признаки:', cat_cols)
print('Числовые признаки:', num_cols)

Категориальные признаки: ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
Числовые признаки: ['powerPS', 'kilometer', 'autoAgeMonths']


3. Разбейте датасет на train val test в отношении 8:1:1

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
indices = rng.permutation(len(X_raw))
n_train, n_val = int(.8 * len(indices)), int(.1 * len(indices))
train_idx, val_idx, test_idx = indices[:n_train], indices[n_train:n_train+n_val], indices[n_train+n_val:]
X_train_raw, X_val_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[val_idx], X_raw.iloc[test_idx]
y_train_raw, y_val_raw, y_test_raw = y_raw[train_idx], y_raw[val_idx], y_raw[test_idx]

def numeric_features(frame):
    values = frame[num_cols].astype(float).copy()
    values['age_sq'] = values['autoAgeMonths'] ** 2
    values['age_log'] = np.log1p(values['autoAgeMonths'])
    values['km_log'] = np.log1p(values['kilometer'])
    values['power_log'] = np.log1p(values['powerPS'])
    values['power_age'] = values['powerPS'] * values['autoAgeMonths']
    return values

def numeric_features(frame):
    values = frame[num_cols].astype(float).copy()
    values['age_sq'] = values['autoAgeMonths'] ** 2
    values['age_log'] = np.log1p(values['autoAgeMonths'])
    values['km_log'] = np.log1p(values['kilometer'])
    values['power_log'] = np.log1p(values['powerPS'])
    values['power_age'] = values['powerPS'] * values['autoAgeMonths']
    return values

levels = {c: sorted(X_train_raw[c].dropna().unique()) for c in cat_cols}
train_numerical = numeric_features(X_train_raw)
num_mean = train_numerical.mean().to_numpy()
num_std = train_numerical.std(ddof=0).to_numpy()
num_std[num_std == 0] = 1
def transform(frame):
    numerical = (numeric_features(frame).to_numpy() - num_mean) / num_std
    categorical = [np.column_stack([(frame[c] == value).astype(float).to_numpy() for value in levels[c]]) for c in cat_cols]
    return np.column_stack([numerical] + categorical)
X_train, X_val, X_test = map(transform, [X_train_raw, X_val_raw, X_test_raw])

y_mean, y_std = y_train_raw.mean(), y_train_raw.std()
y_train = (y_train_raw - y_mean) / y_std
y_val = (y_val_raw - y_mean) / y_std
y_test = (y_test_raw - y_mean) / y_std
print(f'Train: {X_train.shape}; Val: {X_val.shape}; Test: {X_test.shape}')


Train: (800, 208); Val: (100, 208); Test: (100, 208)


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [4]:
search_vgd_const, result_vgd_const = run_search('VGD', 'constant')
all_results.append(result_vgd_const)
display(search_vgd_const)
print(result_vgd_const)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:11: RuntimeWarning: overflow encountered in square
  return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:15: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,5.856654e+07,0.015983,7.434386e+07,400
1,0.00010,5.104807e+07,0.142306,6.721126e+07,400
2,0.00100,2.418282e+07,0.593688,3.761347e+07,400
3,0.01000,1.403405e+07,0.764204,1.803236e+07,400
4,0.10000,9.576667e+06,0.839096,1.915417e+07,400
5,1.00000,inf,-inf,inf,318


{'Метод': 'VGD, eta=1.00e-02', 'Лучший параметр': 0.01, 'Loss_train': 14034047.100766344, 'Loss_val': 18032359.05897374, 'Loss_test': 19861962.399574257, 'R2_train': 0.7642043181094967, 'R2_test': 0.676149023064791, 'Итераций': 400}


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [5]:
search_vgd_decay, result_vgd_decay = run_search('VGD', 'decay')
all_results.append(result_vgd_decay)
display(search_vgd_decay)
print(result_vgd_decay)

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,9.579160e+06,0.839054,1.915058e+07,400
1,0.00010,9.601194e+06,0.838684,1.911842e+07,400
2,0.00100,9.789907e+06,0.835513,1.881427e+07,400
3,0.01000,1.072760e+07,0.819758,1.696028e+07,400
4,0.10000,1.422891e+07,0.760930,1.825523e+07,400
5,1.00000,2.045537e+07,0.656315,3.079331e+07,400


{'Метод': 'VGD, eta(t)=0.1/(1+1.00e-02*t)', 'Лучший параметр': 0.01, 'Loss_train': 10727596.287426649, 'Loss_val': 16960278.798484553, 'Loss_test': 18230191.641025666, 'R2_train': 0.8197582733278934, 'R2_test': 0.7027551833051102, 'Итераций': 400}


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [6]:
search_sgd_const, result_sgd_const = run_search('SGD', 'constant')
all_results.append(result_sgd_const)
display(search_sgd_const)
print(result_sgd_const)

/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:68: RuntimeWarning: overflow encountered in square
  current = np.mean((Xb @ w - y)**2)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:11: RuntimeWarning: overflow encountered in square
  return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:15: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:59: RuntimeWarning: overflow encountered in matmul
  grad_i = Xb[i] * (Xb[i] @ w - y[i])
/var/folders/lz/k7wmmt893152wmtp

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,1.477522e+07,0.751751,1.895599e+07,400
1,0.00010,9.850662e+06,0.834492,1.871297e+07,400
2,0.00100,6.793026e+06,0.885866,2.377869e+07,400
3,0.01000,6.948380e+06,0.883255,2.609919e+07,400
4,0.10000,inf,-inf,inf,78
5,1.00000,NaN,NaN,NaN,1


{'Метод': 'SGD, eta=1.00e-04', 'Лучший параметр': 0.0001, 'Loss_train': 9850661.854338128, 'Loss_val': 18712972.152924445, 'Loss_test': 17409025.975329436, 'R2_train': 0.8344922521394706, 'R2_test': 0.7161443589420088, 'Итераций': 400}


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [7]:
search_sgd_decay, result_sgd_decay = run_search('SGD', 'decay')
all_results.append(result_sgd_decay)
display(search_sgd_decay)
print(result_sgd_decay)

/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:68: RuntimeWarning: overflow encountered in square
  current = np.mean((Xb @ w - y)**2)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:11: RuntimeWarning: overflow encountered in square
  return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:15: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,inf,-inf,inf,78
1,0.00010,inf,-inf,inf,80
2,0.00100,inf,-inf,inf,113
3,0.01000,2.504598e+54,-4.208148e+46,1.134532e+55,400
4,0.10000,6.848435e+12,-1.150643e+05,2.592370e+13,400
5,1.00000,1.786903e+10,-2.992299e+02,2.391353e+10,400


{'Метод': 'SGD, eta(t)=0.1/(1+1.00e+00*t)', 'Лучший параметр': 1.0, 'Loss_train': 17869031340.642033, 'Loss_val': 23913534008.499588, 'Loss_test': 40299179327.356186, 'R2_train': -299.22989088153923, 'R2_test': -656.0815276103447, 'Итераций': 400}


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [8]:
search_sag_const, result_sag_const = run_search('SAG', 'constant')
all_results.append(result_sag_const)
display(search_sag_const)
print(result_sag_const)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:11: RuntimeWarning: overflow encountered in square
  return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:15: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:68: RuntimeWarning: overflow encountered in square
  current = np.mean((Xb @ w - y)**2)


,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,1.477741e+07,7.517146e-01,1.895506e+07,400
1,0.00010,9.852766e+06,8.344569e-01,1.870125e+07,400
2,0.00100,6.771395e+06,8.862291e-01,2.420157e+07,400
3,0.01000,3.290634e+62,-5.528821e+54,3.402397e+62,400
4,0.10000,inf,-inf,inf,398
5,1.00000,inf,-inf,inf,63


{'Метод': 'SAG, eta=1.00e-04', 'Лучший параметр': 0.0001, 'Loss_train': 9852765.968896814, 'Loss_val': 18701251.821558073, 'Loss_test': 17414136.525233354, 'R2_train': 0.8344568994629706, 'R2_test': 0.7160610309935531, 'Итераций': 400}


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [9]:
search_sag_decay, result_sag_decay = run_search('SAG', 'decay')
all_results.append(result_sag_decay)
display(search_sag_decay)
print(result_sag_decay)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:11: RuntimeWarning: overflow encountered in square
  return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))
/var/folders/lz/k7wmmt893152wmtpwytmgmb80000gn/T/ipykernel_11704/1340819829.py:15: RuntimeWarning: overflow encountered in square
  return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)


,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,inf,-inf,inf,398
1,0.00010,inf,-inf,inf,400
2,0.00100,3.038717e+280,-5.105557e+272,3.145613e+280,400
3,0.01000,6.537719e+161,-1.098447e+154,7.456246e+161,400
4,0.10000,4.023714e+97,-6.760518e+89,4.124687e+97,400
5,1.00000,3.378755e+10,-5.666878e+02,3.175225e+10,400


{'Метод': 'SAG, eta(t)=0.1/(1+1.00e+00*t)', 'Лучший параметр': 1.0, 'Loss_train': 33787547942.655323, 'Loss_val': 31752246272.604427, 'Loss_test': 36579596251.95228, 'R2_train': -566.6878415287232, 'R2_test': -595.4334109475598, 'Итераций': 400}


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [10]:
search_momentum_const, result_momentum_const = run_search('Momentum', 'constant')
all_results.append(result_momentum_const)
display(search_momentum_const)
print(result_momentum_const)

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,5.119444e+07,1.398470e-01,6.735510e+07,400
1,0.00010,2.419176e+07,5.935376e-01,3.773425e+07,400
2,0.00100,1.406352e+07,7.637091e-01,1.799328e+07,400
3,0.01000,9.593492e+06,8.388131e-01,1.921594e+07,400
4,0.10000,6.531778e+06,8.902551e-01,2.515647e+07,400
5,1.00000,2.970893e+161,-4.991602e+153,3.594078e+161,400


{'Метод': 'Momentum, eta=1.00e-03', 'Лучший параметр': 0.001, 'Loss_train': 14063521.463007875, 'Loss_val': 17993278.01352649, 'Loss_test': 19898442.89034909, 'R2_train': 0.7637090990687505, 'R2_test': 0.6755542055770306, 'Итераций': 400}


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [11]:
search_momentum_decay, result_momentum_decay = run_search('Momentum', 'decay')
all_results.append(result_momentum_decay)
display(search_momentum_decay)
print(result_momentum_decay)

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,6.533898e+06,0.890219,2.514721e+07,400
1,0.00010,6.552777e+06,0.889902,2.506555e+07,400
2,0.00100,6.724248e+06,0.887021,2.438136e+07,400
3,0.01000,7.707478e+06,0.870501,2.169112e+07,400
4,0.10000,9.831345e+06,0.834817,1.893293e+07,400
5,1.00000,1.373348e+07,0.769254,1.752896e+07,400


{'Метод': 'Momentum, eta(t)=0.1/(1+1.00e+00*t)', 'Лучший параметр': 1.0, 'Loss_train': 13733477.227753496, 'Loss_val': 17528961.381597474, 'Loss_test': 19809963.00324295, 'R2_train': 0.7692543993621767, 'R2_test': 0.6769968776203055, 'Итераций': 400}


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [12]:
search_adam_const, result_adam_const = run_search('Adam', 'constant')
all_results.append(result_adam_const)
display(search_adam_const)
print(result_adam_const)

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,5.707833e+07,0.040988,7.303724e+07,400
1,0.00010,3.990716e+07,0.329492,5.786713e+07,400
2,0.00100,1.154797e+07,0.805975,2.180699e+07,400
3,0.01000,5.670241e+06,0.904730,3.104990e+07,400
4,0.10000,5.528636e+06,0.907110,3.667889e+07,400
5,1.00000,5.528628e+06,0.907110,3.604968e+07,400


{'Метод': 'Adam, eta=1.00e-03', 'Лучший параметр': 0.001, 'Loss_train': 11547966.213655887, 'Loss_val': 21806987.70826509, 'Loss_test': 19817547.212538373, 'R2_train': 0.8059746737169784, 'R2_test': 0.6768732164462385, 'Итераций': 400}


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [13]:
search_adam_decay, result_adam_decay = run_search('Adam', 'decay')
all_results.append(result_adam_decay)
display(search_adam_decay)
print(result_adam_decay)

,parameter,Loss_train,R2_train,Loss_val,iterations
0,0.00001,5.528865e+06,0.907106,3.665804e+07,400
1,0.00010,5.528660e+06,0.907109,3.659233e+07,342
2,0.00100,5.528652e+06,0.907109,3.661100e+07,400
3,0.01000,5.529918e+06,0.907088,3.589477e+07,400
4,0.10000,5.673903e+06,0.904669,3.205379e+07,400
5,1.00000,8.738650e+06,0.853176,2.068077e+07,400


{'Метод': 'Adam, eta(t)=0.1/(1+1.00e+00*t)', 'Лучший параметр': 1.0, 'Loss_train': 8738650.093138753, 'Loss_val': 20680771.25617554, 'Loss_test': 18384127.756186124, 'R2_train': 0.8531759268927034, 'R2_test': 0.7002452419268408, 'Итераций': 400}


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [14]:
summary = pd.DataFrame(all_results).sort_values('Loss_test').reset_index(drop=True)
display(summary)

,Метод,Лучший параметр,Loss_train,Loss_val,Loss_test,R2_train,R2_test,Итераций
0,"SGD, eta=1.00e-04",0.0001,9.850662e+06,1.871297e+07,1.740903e+07,0.834492,0.716144,400
1,"SAG, eta=1.00e-04",0.0001,9.852766e+06,1.870125e+07,1.741414e+07,0.834457,0.716061,400
2,"VGD, eta(t)=0.1/(1+1.00e-02*t)",0.0100,1.072760e+07,1.696028e+07,1.823019e+07,0.819758,0.702755,400
3,"Adam, eta(t)=0.1/(1+1.00e+00*t)",1.0000,8.738650e+06,2.068077e+07,1.838413e+07,0.853176,0.700245,400
4,"Momentum, eta(t)=0.1/(1+1.00e+00*t)",1.0000,1.373348e+07,1.752896e+07,1.980996e+07,0.769254,0.676997,400
5,"Adam, eta=1.00e-03",0.0010,1.154797e+07,2.180699e+07,1.981755e+07,0.805975,0.676873,400
6,"VGD, eta=1.00e-02",0.0100,1.403405e+07,1.803236e+07,1.986196e+07,0.764204,0.676149,400
7,"Momentum, eta=1.00e-03",0.0010,1.406352e+07,1.799328e+07,1.989844e+07,0.763709,0.675554,400
8,"SAG, eta(t)=0.1/(1+1.00e+00*t)",1.0000,3.378755e+10,3.175225e+10,3.657960e+10,-566.687842,-595.433411,400
9,"SGD, eta(t)=0.1/(1+1.00e+00*t)",1.0000,1.786903e+10,2.391353e+10,4.029918e+10,-299.229891,-656.081528,400


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

1. SGD с шагом 0.0001, так как в таблице у него минимальный Loss_test и максимальный R²_test 0.7161
2. R²_train показывает качество на данных обучения, а R²_test качество на новых данных.
3. Высокий R²_train означает, что модель хорошо обучилась, а высокий R²_test что она может применять полученные закономерности к новым данным. Если R²_train намного больше R²_test это говорит о переобучении.